In [1]:
import gc
import re
from pathlib import Path
import polars as pl

# -----------------------------------------------------------------------------
# HÀM XỬ LÝ CHUỖI NGÀY THÁNG (ĐÃ TỐI ƯU & BẢO VỆ)
# -----------------------------------------------------------------------------
def parse_multi_date(col_name: str) -> pl.Expr:
    col_expr = pl.col(col_name)
    
    # 1. Chuẩn hóa chuỗi (chỉ áp dụng nếu là chuỗi)
    clean_str = (
        col_expr
        .cast(pl.String)
        .str.strip_chars()
        .str.replace_all("/", "-")
    )

    # 2. Parse đa định dạng
    parsed_datetime = pl.coalesce([
        # Nếu cột vốn đã là Datetime/Date thì giữ nguyên
        col_expr.cast(pl.Datetime, strict=False),
        
        # Parse chuỗi dạng Năm - Ngày - Tháng
        clean_str.str.to_datetime("%Y-%d-%m %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m", strict=False),

        # Parse chuỗi dạng Năm - Tháng - Ngày
        clean_str.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d", strict=False),

        # Parse chuỗi dạng Ngày - Tháng - Năm (Việt Nam)
        clean_str.str.to_datetime("%d-%m-%Y %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y %H:%M", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y", strict=False),
    ])

    return parsed_datetime

In [ ]:
# -----------------------------------------------------------------------------
# 1. CẤU HÌNH ĐƯỜNG DẪN & CỘT CẦN LẤY
# -----------------------------------------------------------------------------
path = Path(r"C:\Users\Win 10\Desktop\streamlit")

selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "tg_quydinh", "ngay_gui_bp",
    "time_pcp", "tg_quydinhphat", "tg_chenhlechphat", "ma_dv_viettel", "ma_buucuc_goc",
    "ma_buucuc_phat", "ma_trangthai", "ma_doitac", "ma_khgui", "tg_toantrinh",
    "danhgia_time_gach_bp1", "danhgia_time_gach_bp2", "danhgia_time_gach_bp3",
    "time_gach_bp", "time_gach_bp2", "time_gach_bp3", "tg_ptc", "tg_nhantai_bcp", "trong_luong","KHAU_SAI",
    "tien_cod",	"tienhang",	"tong_cuoc"

]

date_cols = [
    "ngay_gui_bp", "tg_ptc", "tg_nhantai_bcp", "time_pcp", 
    "tg_quydinhphat", "time_gach_bp", "time_gach_bp2", "time_gach_bp3"
]

# -----------------------------------------------------------------------------
# 2. NẠP VÀ TỐI ƯU TRỰC TIẾP TỪNG FILE EXCEL
# -----------------------------------------------------------------------------
print("🚀 1/4. Đang nạp và xử lý từng file Excel...")

dfs = []
excel_files = list(path.glob("202608_*.xlsx"))

if not excel_files:
    print("❌ Không tìm thấy file Excel nào khớp với mẫu!")
else:
    for file in excel_files:
        # Đọc file với mẫu 10,000 dòng để đạt tốc độ cao nhất
        df = pl.read_excel(file, drop_empty_rows=False, infer_schema_length=10000)
        
        # Chỉ giữ lại các cột cần thiết ngay từ đầu để tiết kiệm RAM
        existing_cols = [c for c in selected_columns if c in df.columns]
        df = df.select(existing_cols)
        
        # Parse ngày trực tiếp trên từng file nhỏ (Truyền c dạng chuỗi str)
        date_exprs = [parse_multi_date(c).alias(c) for c in date_cols if c in df.columns]
        if date_exprs:
            df = df.with_columns(date_exprs)
            
        df = df.with_columns(pl.lit(file.name).alias("Ten_File_Nguon"))
        dfs.append(df)

    print("⚡ 2/4. Đang gộp các bảng dữ liệu...")
    df_final = pl.concat(dfs, how="diagonal_relaxed")
    
    del dfs
    gc.collect()

# -----------------------------------------------------------------------------
# 3. TRÍCH XUẤT VÀ CHUẨN HÓA DỮ LIỆU
# -----------------------------------------------------------------------------
print("🛠️ 3/4. Đang trích xuất thông tin...")

# Ghép tuyến
df_final = df_final.with_columns(
    pl.concat_str([pl.col("tinh_nhan"), pl.lit("->"), pl.col("tinh_phat")]).alias("tuyen")
)

# Trích xuất số từ tg_chenhlechphat
df_final = df_final.with_columns(
    pl.col("tg_chenhlechphat")
    .cast(pl.String)
    .str.extract(r"(-?\d+)", 1)
    .cast(pl.Int64, strict=False)
    .alias("tg_chenhlechphat_so")
)

# -----------------------------------------------------------------------------
# 4. TÍNH LOGIC VÀ PHÂN LOẠI
# -----------------------------------------------------------------------------
print("🧠 4/4. Đang tính toán cờ KPI và phân loại giao hàng...")

df_final = (
    df_final.with_columns([
        # Ngày bắt đầu phải phát
        pl.when(pl.col("tg_nhantai_bcp").is_not_null())
        .then(pl.col("tg_nhantai_bcp"))
        .when(pl.col("time_pcp").is_not_null())
        .then(pl.col("time_pcp"))
        .otherwise(pl.col("ngay_gui_bp"))
        .alias("ngay_bat_dau_phai_phat"),

        # Ngày phát cuối cùng
        pl.when(pl.col("tg_ptc").is_not_null())
        .then(pl.col("tg_ptc"))
        .otherwise(
            pl.max_horizontal([
                pl.col("time_gach_bp"),
                pl.col("time_gach_bp2"),
                pl.col("time_gach_bp3"),
            ])
        )
        .alias("ngay_phat_cuoi_cung"),

        # Cờ PTC
        pl.when(pl.col("tg_ptc").is_not_null()).then(1).otherwise(0).alias("PTC"),

        # Cờ PTC_1
        pl.when(
            (pl.col("tg_ptc") == pl.col("time_gach_bp"))
            & (pl.col("danhgia_time_gach_bp1") == "Đúng chỉ tiêu")
        )
        .then(1)
        .otherwise(0)
        .alias("PTC_1"),

        # Đánh giá giao hàng
        pl.when(pl.col("tg_chenhlechphat_so").is_null())
        .then(pl.lit("Không xác định"))
        .when(pl.col("tg_chenhlechphat_so") > 0)
        .then(pl.lit("Giao không đúng giờ"))
        .otherwise(pl.lit("Giao đúng giờ"))
        .alias("danh_gia_giao_hang"),
    ])
    .filter(pl.col("ngay_bat_dau_phai_phat").is_not_null())
)

print("=" * 70)
print(f"✅ HOÀN THÀNH! Tổng số bản ghi đã xử lý: {df_final.height:,}")
print("=" * 70)

🚀 1/4. Đang nạp và xử lý từng file Excel...


Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 33, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 37, falling back to string
Could not determine dtype for column 38, falling back to string
Could not determine dtype for column 39, falling back to string
C:\Users\Win 10\AppData\Local\Temp\ipykernel_1340\2983986765.py:34: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(file, drop_empty_rows=False, infer_schema_length=10000)


In [ ]:
# -----------------------------------------------------------------------------
# TỔNG HỢP VÀ EXPLODE KHOẢNG NGÀY
# -----------------------------------------------------------------------------
selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat",
    "ma_trangthai", "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang",
    "trong_luong", "Ten_File_Nguon","tg_ptc","KHAU_SAI","tien_cod",	"tienhang",	"tong_cuoc"
]

existing_cols = [col for col in selected_columns if col in df_final.columns]
df_raw = df_final.select(existing_cols)
df_exploded = df_raw
# df_exploded = (
#     df_raw.filter(
#         pl.col("ngay_bat_dau_phai_phat").is_not_null()
#         & pl.col("ngay_phat_cuoi_cung").is_not_null()
#         & (pl.col("ngay_phat_cuoi_cung") >= pl.col("ngay_bat_dau_phai_phat"))
#     )
#     .with_columns(
#         pl.date_ranges(
#             start=pl.col("ngay_bat_dau_phai_phat"),
#             end=pl.col("ngay_phat_cuoi_cung"),
#             interval="1d"
#         ).alias("ngay_trong_khoang")
#     )
#     .explode("ngay_trong_khoang")
# )

In [ ]:
file_path = "vtp_odr_assessment_aug_2026.csv"

# Đọc file bằng Polars
df = pl.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên để kiểm tra
df.head(5)

In [4]:
df_phat_processed = df_exploded.with_columns([
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 500)
    .then(pl.lit("< 500g"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 1000)
    .then(pl.lit("500g - < 1000g"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 2000)
    .then(pl.lit("1000g - < 2000g"))
    .otherwise(pl.lit(">= 2000g"))
    .alias("nhom_trong_luong")
])

index_cols = [
    "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat", "ma_trangthai","ma_phieugui",
    "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang","KHAU_SAI",
    # "ngay_trong_khoang",
    "nhom_trong_luong", "Ten_File_Nguon","tg_ptc", "tien_cod",	"tienhang",	"trong_luong",	"tong_cuoc"
]

cols_exist = [c for c in index_cols if c in df_phat_processed.columns]
df_phat_processed = df_phat_processed.select(cols_exist)

In [5]:
df_phat_processed = df_phat_processed.filter(pl.col("ma_trangthai") == 501)
df_phat_processed = df_phat_processed.filter(pl.col("tg_ptc").is_not_null())
df_phat_processed

tinh_phat,ma_dv_viettel,ma_buucuc_phat,ma_trangthai,ma_phieugui,ma_doitac,ma_khgui,tuyen,PTC,PTC_1,ngay_bat_dau_phai_phat,ngay_phat_cuoi_cung,danh_gia_giao_hang,KHAU_SAI,nhom_trong_luong,Ten_File_Nguon,tg_ptc,tien_cod,tienhang,trong_luong,tong_cuoc
str,str,str,i64,str,str,str,str,i32,i32,datetime[μs],datetime[μs],str,str,str,str,datetime[μs],i64,i64,i64,i64
"""TNN""","""NDD""","""HBTNDTY""",501,"""VTPVN9043819815""","""VTPVN""","""XNLI1""","""BNH->TNN""",1,0,2026-07-31 05:55:40,2026-08-05 11:48:16,"""Giao không đúng giờ""","""FM""","""1000g - < 2000g""","""202608_1-12.xlsx""",2026-08-05 11:48:16,53399,59999,1000,14820
"""HNI""","""NDD""","""GLM""",501,"""VTPVN9049368714""","""VTPVN""","""DBG2689""","""BNH->HNI""",1,0,2026-07-31 07:06:40,2026-08-05 09:38:58,"""Giao không đúng giờ""","""LM""","""500g - < 1000g""","""202608_1-12.xlsx""",2026-08-05 09:38:58,1462500,1950000,500,14535
"""PHO""","""NDD""","""HBVPTD""",501,"""VTPVN9047869014""","""VTPVN""","""BNH5415""","""BNH->PHO""",1,0,2026-07-31 12:11:44,2026-08-04 16:26:18,"""Giao không đúng giờ""","""LM""","""< 500g""","""202608_1-12.xlsx""",2026-08-04 16:26:18,206121,205821,150,14535
"""HNI""","""NDD""","""HN04""",501,"""VTPVN9048348917""","""VTPVN""","""HBI6248""","""HNI->HNI""",1,0,2026-07-31 05:05:19,2026-08-03 19:07:51,"""Giao không đúng giờ""","""LM""","""< 500g""","""202608_1-12.xlsx""",2026-08-03 19:07:51,18000,18000,170,10640
"""PHO""","""NDD""","""HBPTPH""",501,"""VTPVN9049139520""","""VTPVN""","""SSNDPC8""","""HNI->PHO""",1,0,2026-07-31 06:36:57,2026-08-02 14:51:05,"""Giao đúng giờ""","""LM""",""">= 2000g""","""202608_1-12.xlsx""",2026-08-02 14:51:05,1074400,1360000,12000,67830
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HNI""","""TTNEW""","""QGG""",501,"""VTPVN9047295855""","""VTPVN""","""HBLACD4745""","""LAN->HNI""",1,0,2026-08-29 13:21:42,2026-08-31 14:57:14,"""Giao không đúng giờ""","""LM""","""500g - < 1000g""","""202608_25-31.xlsx""",2026-08-31 14:57:14,0,58999,500,17765
"""HNI""","""TTNEW""","""PUT""",501,"""VTPVN9047608953""","""VTPVN""","""HBCTAK9382""","""CTO->HNI""",1,1,2026-08-31 05:16:03,2026-08-31 12:12:27,"""Giao đúng giờ""","""FM""","""500g - < 1000g""","""202608_25-31.xlsx""",2026-08-31 12:12:27,0,393000,500,17765
"""HNI""","""TTNEW""","""HN01""",501,"""VTPVN9043247157""","""VTPVN""","""VKVMN101""","""BDG->HNI""",1,1,2026-08-31 12:18:32,2026-08-31 14:33:17,"""Giao đúng giờ""","""FM""","""500g - < 1000g""","""202608_25-31.xlsx""",2026-08-31 14:33:17,0,125000,500,17765


In [6]:
# 1. Định nghĩa đường dẫn chính xác tới thư mục Tikok_VTP trên Google Drive
drive_folder = Path(r"G:\My Drive\Tikok_VTP")

# Tự tạo thư mục nếu chưa tồn tại
drive_folder.mkdir(parents=True, exist_ok=True)

# 2. Đặt tên file Parquet
parquet_path = drive_folder / "TTS_phat_data.parquet"

# 3. Ghi file từ Polars ra Parquet
df_phat_processed.write_parquet(parquet_path, compression="snappy")

print("=" * 70)
print(f"✅ Đã xuất thành công file Parquet: {parquet_path}")
print("☁️ Google Drive đang tự động đồng bộ file này lên cloud!")
print("=" * 70)

✅ Đã xuất thành công file Parquet: G:\My Drive\Tikok_VTP\TTS_phat_data.parquet
☁️ Google Drive đang tự động đồng bộ file này lên cloud!
